# Notebook 08 — Controlled Auxiliary-Task Ablation and Screening-to-Potency Transfer

**Scientific question:** Do related CYP screening, Emax, and time-dependent-inhibition (TDI) assays improve direct-inhibition pIC50 predictions? If so, is the benefit attributable to auxiliary labels on existing molecules, additional auxiliary-only structures, or using *cross-fitted, structure-predicted* screening responses as intermediate features?

Two independent experiments use the **same five compound/scaffold folds** as Notebooks 05–07:

1. **Controlled frozen-feature multi-task ablations:** direct-only; direct + screening; direct + Emax; direct + TDI; direct + all assays (on the same primary compounds); then screening and all assays with eligible auxiliary-only compounds. Architecture, optimizer, validation and task weights are held fixed.
2. **Screening-to-potency transfer:** predict the four single-concentration `log2fc_estimate` responses from molecular features using an **inner scaffold cross-fit**, then train a LightGBM pIC50 model using only these *predicted* responses and molecular descriptors. Compare against LightGBM trained on the same original features without screening predictions.

**Leakage control:** For an outer fold, *every assay measurement* belonging to a held-out molecule **or its scaffold group** is excluded from training. Inner development groups are also removed from auxiliary training. For the screening-to-pIC50 approach, the secondary model is fitted using **out-of-fold predictions** from the first stage, not its in-sample fits. At inference the inputs are predicted screening responses, never experimental responses measured for held-out compounds.

**Important:** The single-concentration screen used a different protocol from direct inhibition. It is treated as a related but non-equivalent assay. There is no assumption that `log2fc_estimate` is pIC50, or that its sign directly indicates a mechanistic mode of inhibition. Our interval-excess diagnostic is **not the official challenge MA-ST-RAE metric**.

The notebook is designed for Google Colab. **Run in `FAST_MODE=True` first**, then change to `False` and use Google Drive storage for the full five-fold experiment. Frozen CheMeleon embeddings require a trusted, author-released checkpoint and can be cached; a mock-feature option exists solely for synthetic execution tests.

## 0 · Google Colab dependencies

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.6" "scipy>=1.11" "torch>=2.3" "chemprop>=2.2.0,<2.4" pandas matplotlib

## 1 · Configuration and imports

In [ ]:
from pathlib import Path
from urllib.request import urlopen, urlretrieve
from collections import Counter
import os, io, json, time, zipfile, hashlib, platform, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb
import torch
from torch import nn as tnn
from torch.utils.data import DataLoader, TensorDataset
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

# Run one outer fold and lighter training first. Set False for all five folds.
FAST_MODE = True
FOLDS_TO_RUN = [0] if FAST_MODE else list(range(5))
SEED = 2026
N_SPLITS = 5
INNER_FOLDS = 2 if FAST_MODE else 3
RUN_ABLATIONS = True
RUN_SCREEN_STACK = True
# Select a subset of arms here if Colab runtime is limited.
ABLATION_ARMS = ['direct_only', 'screen_only', 'emax_only', 'tdi_only',
                 'all_primary', 'screen_plus_extra', 'all_plus_extra']
MAX_EPOCHS = 12 if FAST_MODE else 45
PATIENCE = 4 if FAST_MODE else 8
BATCH_SIZE = 128
HIDDEN = 128
DROPOUT = 0.30
LR = 1.5e-4
WEIGHT_DECAY = 0.005
N_LGBM_TREES = 60 if FAST_MODE else 220
USE_CHEMELEON = True
# Synthetic execution tests only. Never publish mock-embedding predictions.
MOCK_EMBEDDINGS = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ISOFORMS = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
# Optional local CSV overrides for development/tests.
DATA_OVERRIDES = {}  # {'direct':'/content/direct.csv','single':'/content/screen.csv',...}
# Optionally mount Google Drive first; use a distinct folder for any changed configuration.
# from google.colab import drive; drive.mount('/content/drive')
PERSISTENT_ROOT = None  # e.g. '/content/drive/MyDrive/CYP_Notebook08'
# Optional previous Notebook 05/06/07 result ZIPs to audit matching outer folds.
PRIOR_RESULT_ZIPS = []  # e.g. ['/content/cyp_notebook_05_results.zip']
# Reuse compatible Notebook 07 feature cache by pointing to its cache directory.
SHARED_FEATURE_CACHE = None  # e.g. '/content/drive/MyDrive/CYP_Notebook07/cyp_notebook_07_cache'
EXPERIMENT_TAG = 'v1'  # Increment when changing methodology to avoid stale cached predictions.
ROOT_URL = 'https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'
FILES = {'direct':'cyp-challenge-TRAIN_inhibition.csv',
         'single':'cyp-challenge-single-concentration-TRAIN.csv',
         'emax':'cyp-challenge-TRAIN_Emax.csv',
         'tdi':'cyp-challenge-TRAIN_TDI.csv'}
BASE = Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
OUT = BASE/'cyp_notebook_08_results'
CACHE_DIR = BASE/'cyp_notebook_08_cache'
OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Runtime:', platform.python_version(), 'Torch:',torch.__version__,
      'RDKit:',rdBase.rdkitVersion,'LightGBM:',lgb.__version__, 'Device:',DEVICE)
print('Mode:', 'FAST, one fold' if FAST_MODE else 'FULL, five folds')

## 2 · Load datasets, canonicalize compounds, reproduce the exact outer folds

In [ ]:
def fetch_source(key):
    if key in DATA_OVERRIDES:
        raw=Path(DATA_OVERRIDES[key]).read_bytes()
    else:
        with urlopen(ROOT_URL+FILES[key],timeout=180) as response:
            raw=response.read()
    return pd.read_csv(io.BytesIO(raw)),hashlib.sha256(raw).hexdigest()

sources={}; source_shas={}
for key in FILES:
    sources[key], source_shas[key] = fetch_source(key)
    print(key, sources[key].shape, 'SHA256:', source_shas[key][:16])

primary=sources['direct'].copy()
def canon_and_group(s):
    if not isinstance(s,str) or not s.strip(): return None,None
    mol=Chem.MolFromSmiles(s)
    if mol is None: return None,None
    canonical=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(mol)
    group=(Chem.MolToSmiles(core,canonical=True,isomericSmiles=False)
           if core.GetNumAtoms() else 'ACYCLIC_'+canonical)
    return canonical,group

if 'SMILES' not in primary.columns:
    raise KeyError('Direct-inhibition table must contain SMILES')
struct=[canon_and_group(s) for s in primary.SMILES]
primary['canonical_smiles']=[a for a,b in struct]
primary['scaffold_group']=[b for a,b in struct]
assert primary.canonical_smiles.notna().all(), 'Resolve invalid primary SMILES before proceeding'
assert primary.canonical_smiles.is_unique, 'Resolve duplicate primary structures before CV'
primary=primary.reset_index(drop=True)
for iso in ISOFORMS:
    assert f'{iso}_pIC50_direct_inhibition' in primary, iso

fold_ids=np.full(len(primary),-1,dtype=int)
for fold, (_,vi) in enumerate(GroupKFold(n_splits=N_SPLITS).split(primary,groups=primary.scaffold_group)):
    fold_ids[vi]=fold
primary['fold']=fold_ids
manifest=primary[['canonical_smiles','scaffold_group','fold']].copy()
manifest.to_csv(OUT/'fold_manifest.csv',index=False)
assert (fold_ids>=0).all() and set(fold_ids)==set(range(N_SPLITS))
assert not manifest.groupby('scaffold_group').fold.nunique().gt(1).any()

def read_result_zip(path,name):
    with zipfile.ZipFile(path) as z:
        choices=[p for p in z.namelist() if p==name or p.endswith('/'+name)]
        if len(choices)!=1:raise ValueError(f'Expected one {name} in {path}; found {choices}')
        return z.read(choices[0])
for z in PRIOR_RESULT_ZIPS:
    prev=pd.read_csv(io.BytesIO(read_result_zip(z,'fold_manifest.csv')))
    lhs=manifest.set_index('canonical_smiles').fold
    rhs=prev.set_index('canonical_smiles').fold
    assert rhs.index.is_unique and set(rhs.index)==set(lhs.index)
    assert np.array_equal(rhs.loc[lhs.index].to_numpy(),lhs.to_numpy()),f'Fold mismatch: {z}'
    print('Matched identities and folds to',z)
print('Primary molecules:',len(primary),'fold sizes:',np.bincount(fold_ids))

## 3 · Map the verified screening columns and group auxiliary tasks by experimental family

In [ ]:
# Unlike Notebook 07, these screening column names are explicitly confirmed by the dataset.
# The synthetic test fixture uses equivalent alternate names.
AUDIT=[]
aux_tables={}
primary_id_lookup=dict(zip(primary.Molecule_Name.astype(str),primary.canonical_smiles)) if 'Molecule_Name' in primary else {}

def attach_identity(df,source):
    f=df.copy()
    if 'SMILES' in f:
        f['canonical_smiles']=[canon_and_group(s)[0] for s in f['SMILES']]
    elif 'Molecule_Name' in f:
        f['canonical_smiles']=f.Molecule_Name.astype(str).map(primary_id_lookup)
        warnings.warn(f'{source}: no SMILES. Cannot recover auxiliary-only molecules by ID.')
    else: raise ValueError(f'{source} lacks SMILES and Molecule_Name')
    bad=int(f.canonical_smiles.isna().sum())
    if bad:print(source,'invalid/missing structures:',bad)
    return f.dropna(subset=['canonical_smiles'])

aux_sources={k:attach_identity(sources[k],k) for k in ['single','emax','tdi']}

def add_task(family,iso,col,f,kind='reg'):
    vals=pd.to_numeric(f[col].replace({True:1,False:0,'True':1,'False':0}),errors='coerce')
    tab=pd.DataFrame({'structure':f.canonical_smiles,'value':vals}).dropna()
    if kind=='binary':
        tab=tab[tab.value.isin([0,1])]
        g=tab.groupby('structure').value.agg(['min','max'])
        conflict=g['min']!=g['max']
        if conflict.any():print('Conflicting binary labels dropped:',int(conflict.sum()),col)
        series=g.loc[~conflict,'min'].astype(float)
    else: series=tab.groupby('structure').value.mean()
    if len(series)==0: return
    name=f'{family}:{iso}:{col}'
    aux_tables[name]=dict(kind=kind,family=family,isoform=iso,series=series)
    AUDIT.append(dict(task=name,source=family,kind=kind,isoform=iso,column=col,n=len(series)))

screen=aux_sources['single']
SCREEN_ISO_COL='enzyme' if 'enzyme' in screen else 'CYP' if 'CYP' in screen else None
SCREEN_VALUE_COL=('log2fc_estimate' if 'log2fc_estimate' in screen
                  else 'log2_fold_change' if 'log2_fold_change' in screen else None)
if not SCREEN_ISO_COL or not SCREEN_VALUE_COL:
    raise KeyError(f'Screen schema mismatch. Columns: {list(screen.columns)}')
print('Screen column mapping:',SCREEN_ISO_COL,'→ CYP;',SCREEN_VALUE_COL,'→ response')
for iso in ISOFORMS:
    sub=screen.loc[screen[SCREEN_ISO_COL].astype(str).str.upper().str.strip()==iso]
    add_task('screen',iso,SCREEN_VALUE_COL,sub)

# Emax: retain direct and TDI-condition Emax as two distinct regression endpoints.
emax=aux_sources['emax']
for iso in ISOFORMS:
    cs=[c for c in emax.columns if str(c).startswith(iso+'_') and 'emax' in c.lower()
        and not any(v in c.lower() for v in ['_conf_', '_std', '_error','_sem','_se'])]
    for c in cs: add_task('emax',iso,c,emax)

# TDI condition pIC50 (separate from direct inhibition) and binary TDI only for 2D6/3A4.
tdi=aux_sources['tdi']
for iso in ISOFORMS:
    cs=[c for c in tdi.columns if c.startswith(iso+'_')
        and 'pic50' in c.lower() and 'tdi' in c.lower() and 'direct' not in c.lower()
        and not any(v in c.lower() for v in ['_conf_', '_std','_error','_sem','_se'])]
    for c in cs: add_task('tdi',iso,c,tdi)
    if iso in ('CYP2D6','CYP3A4'):
        for c in [f'{iso}_is_TDI']:
            if c in tdi: add_task('tdi',iso,c,tdi,kind='binary')

AUDIT=pd.DataFrame(AUDIT)
AUDIT.to_csv(OUT/'auxiliary_column_audit.csv',index=False)
display(AUDIT)
if len(AUDIT[AUDIT.source=='screen'])!=4:
    raise ValueError('Expected exactly four isoform-specific screening regression tasks')
print('Mapped auxiliary tasks:',len(aux_tables))

## 4 · Construct a union of labeled structures; inspect label coverage

In [ ]:
all_canon=list(primary.canonical_smiles)
extra=sorted(set().union(*(set(v['series'].index) for v in aux_tables.values()))-set(all_canon))
all_canon+=extra
idx_map={s:i for i,s in enumerate(all_canon)}
N_PRIMARY=len(primary);N_ALL=len(all_canon)
all_groups=np.asarray([canon_and_group(s)[1] for s in all_canon],dtype=object)
assert len(set(all_canon))==N_ALL and all(g is not None for g in all_groups)
reg_aux=sorted(k for k,v in aux_tables.items() if v['kind']=='reg')
binary_aux=sorted(k for k,v in aux_tables.items() if v['kind']=='binary')
reg_tasks=['direct:'+s for s in ISOFORMS]+reg_aux
bin_tasks=binary_aux
YR=np.full((N_ALL,len(reg_tasks)),np.nan,dtype=np.float32)
YB=np.full((N_ALL,len(bin_tasks)),np.nan,dtype=np.float32)
for j,iso in enumerate(ISOFORMS):
    YR[:N_PRIMARY,j]=pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition'],errors='coerce').to_numpy(np.float32)
for j,key in enumerate(reg_aux,start=4):
    for smiles,value in aux_tables[key]['series'].items(): YR[idx_map[smiles],j]=value
for j,key in enumerate(bin_tasks):
    for smiles,value in aux_tables[key]['series'].items(): YB[idx_map[smiles],j]=value
assert not np.isfinite(YR[N_PRIMARY:,:4]).any()
COVERAGE=pd.DataFrame({'task':reg_tasks+bin_tasks,
                       'kind':['reg']*len(reg_tasks)+['binary']*len(bin_tasks),
                       'n_labeled':list(np.isfinite(YR).sum(axis=0))+list(np.isfinite(YB).sum(axis=0))})
COVERAGE.to_csv(OUT/'task_coverage.csv',index=False)
display(COVERAGE)
print('Primary compounds',N_PRIMARY,'extra auxiliary-only compounds',len(extra),'union',N_ALL)

## 5 · Molecular features: RDKit 2D + Morgan counts, optionally frozen CheMeleon

In [ ]:
# Use the same CheMeleon author-released checkpoint/interface as Notebook 07.
# Any cached features are keyed by the complete ordered molecule list and RDKit version.
from rdkit import DataStructs
mol_hash=hashlib.sha256('\n'.join(all_canon).encode()).hexdigest()[:16]
feature_file=CACHE_DIR/f'features_{mol_hash}_{rdBase.rdkitVersion}.npz'
shared_file=(Path(SHARED_FEATURE_CACHE)/feature_file.name) if SHARED_FEATURE_CACHE else None
feature_source=shared_file if shared_file and shared_file.exists() else feature_file
if feature_source.exists():
    z=np.load(feature_source)
    desc=z['rdkit'].astype(np.float32)
    count=z['morgan'].astype(np.float32)
    descriptor_names=list(z['descriptor_names'])
    print('Loaded descriptors from',feature_source)
else:
    descriptor_names=[name for name,fn in Descriptors._descList]
    desc=np.full((N_ALL,len(descriptor_names)),np.nan,dtype=np.float32)
    count=np.zeros((N_ALL,2048),dtype=np.float32)
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048,countSimulation=False)
    functions=[fn for name,fn in Descriptors._descList]
    for i,s in enumerate(all_canon):
        mol=Chem.MolFromSmiles(s)
        count[i]=np.asarray(gen.GetCountFingerprintAsNumPy(mol),dtype=np.float32)
        for j,fn in enumerate(functions):
            try:desc[i,j]=float(fn(mol))
            except Exception:pass
        if i%2000==0:print('Descriptor feature',i,'/',N_ALL)
    desc[~np.isfinite(desc)]=np.nan
    np.savez_compressed(feature_file,rdkit=desc,morgan=count,
                        descriptor_names=np.array(descriptor_names))
assert len(desc)==len(count)==N_ALL

emb=np.zeros((N_ALL,0),dtype=np.float32)
if USE_CHEMELEON:
    emb_file=CACHE_DIR/f'chemeleon_{mol_hash}_{"MOCK" if MOCK_EMBEDDINGS else "PRETRAINED"}.npy'
    shared_emb=Path(SHARED_FEATURE_CACHE)/emb_file.name if SHARED_FEATURE_CACHE else None
    if shared_emb and shared_emb.exists():
        emb=np.load(shared_emb)
        print('Loaded frozen embeddings from',shared_emb)
    elif emb_file.exists():
        emb=np.load(emb_file)
        print('Loaded frozen embeddings from',emb_file)
    elif MOCK_EMBEDDINGS:
        # Non-scientific synthetic test only. Do not report these as pretrained embeddings.
        rng=np.random.default_rng(SEED)
        emb=(count @ rng.normal(size=(count.shape[1],32)).astype(np.float32)
             /np.sqrt(count.shape[1])).astype(np.float32)
        np.save(emb_file,emb)
    else:
        from chemprop import data, featurizers, models, nn
        from chemprop.data import BatchMolGraph
        checkpoint_path=CACHE_DIR/'chemeleon_mp.pt'
        if not checkpoint_path.exists():
            print('Retrieving trusted CheMeleon message-passing checkpoint from Zenodo...')
            urlretrieve('https://zenodo.org/records/15460715/files/chemeleon_mp.pt',checkpoint_path)
        # Restricted mode for downloaded checkpoints; never set weights_only=False here.
        checkpoint=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
        mp=nn.BondMessagePassing(**checkpoint['hyper_parameters'])
        mp.load_state_dict(checkpoint['state_dict'])
        encoder=models.MPNN(message_passing=mp,agg=nn.MeanAggregation(),
                   predictor=nn.RegressionFFN(input_dim=mp.output_dim),batch_norm=False).to(DEVICE)
        encoder.eval()
        featurizer=featurizers.SimpleMoleculeMolGraphFeaturizer()
        batches=[]
        for start in range(0,N_ALL,64):
            bg=BatchMolGraph([featurizer(Chem.MolFromSmiles(s)) for s in all_canon[start:start+64]])
            bg.to(device=encoder.device)
            with torch.inference_mode():
                part=encoder.fingerprint(bg).detach().cpu().numpy().astype(np.float32)
            batches.append(part)
            if start%1280==0:print('Embedded',min(start+64,N_ALL),'/',N_ALL)
        emb=np.vstack(batches)
        np.save(emb_file,emb)
        del encoder,mp
        if torch.cuda.is_available():torch.cuda.empty_cache()
    if emb.shape[0]!=N_ALL or not np.isfinite(emb).all():
        raise ValueError('Corrupt or incompatible pretrained embedding cache')

X_RAW=np.concatenate([emb,desc,count],axis=1).astype(np.float32)
assert X_RAW.shape[0]==N_ALL
print('Raw features:',X_RAW.shape,'CheMeleon dim:',emb.shape[1],
      'RDKit descriptor dim:',desc.shape[1],'Morgan count dim:',count.shape[1])

## 6 · Leakage-safe splits, metrics, resumable checkpoints

In [ ]:
setting_fields=dict(source_shas=source_shas,seed=SEED,experiment_tag=EXPERIMENT_TAG,
    fold_hash=hashlib.sha256(manifest.to_csv(index=False).encode()).hexdigest(),
    screening_col=SCREEN_VALUE_COL,reg_tasks=reg_tasks,bin_tasks=bin_tasks,
    feature_hash=mol_hash,embedding_dim=emb.shape[1],mock=MOCK_EMBEDDINGS,
    max_epochs=MAX_EPOCHS,patience=PATIENCE,lr=LR,hidden=HIDDEN,dropout=DROPOUT,
    weight_decay=WEIGHT_DECAY,batch_size=BATCH_SIZE,inner_folds=INNER_FOLDS,
    n_lgbm_trees=N_LGBM_TREES)
RUN_SIGNATURE=hashlib.sha256(json.dumps(setting_fields,sort_keys=True).encode()).hexdigest()[:16]
PARTIAL=OUT/'partials'/RUN_SIGNATURE;PARTIAL.mkdir(parents=True,exist_ok=True)
print('Run signature',RUN_SIGNATURE)

ALL_G=np.asarray(all_groups,dtype=object)
PRIMARY_G=ALL_G[:N_PRIMARY]
def outer_split(fold):
    test=np.flatnonzero(fold_ids==fold)
    train=np.flatnonzero(fold_ids!=fold)
    test_groups=set(PRIMARY_G[test])
    eligible=np.array([i for i in range(N_ALL) if ALL_G[i] not in test_groups],dtype=int)
    assert not test_groups.intersection(ALL_G[eligible])
    assert set(train).issubset(set(eligible))
    return train,test,eligible

def dev_split(fold,outer_train):
    splitter=GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED+fold)
    ii,jj=next(splitter.split(outer_train,groups=PRIMARY_G[outer_train]))
    train=outer_train[ii];dev=outer_train[jj]
    assert not set(PRIMARY_G[train]).intersection(PRIMARY_G[dev])
    return train,dev

def metrics(y,p,lo=None,hi=None):
    y=np.asarray(y,float);p=np.asarray(p,float)
    ok=np.isfinite(y)&np.isfinite(p)
    result=dict(n=int(ok.sum()),mae=float(mean_absolute_error(y[ok],p[ok])) if ok.any() else np.nan,
                r2=float(r2_score(y[ok],p[ok])) if ok.sum()>1 else np.nan,
                spearman=float(spearmanr(y[ok],p[ok]).statistic) if ok.sum()>2 else np.nan)
    result['mean_interval_excess']=np.nan
    if lo is not None and hi is not None and ok.any():
        lo=np.asarray(lo,float);hi=np.asarray(hi,float)
        good=ok&np.isfinite(lo)&np.isfinite(hi)&(hi>=lo)
        if good.any():
            exc=np.maximum(lo[good]-p[good],0)+np.maximum(p[good]-hi[good],0)
            result['mean_interval_excess']=float(np.mean(exc))
    return result

CI_LOW=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition_conf_low'],
                errors='coerce').to_numpy(float) for iso in ISOFORMS])
CI_HIGH=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition_conf_high'],
                errors='coerce').to_numpy(float) for iso in ISOFORMS])

def partial_path(name,fold):return PARTIAL/f'{name}_fold{fold}.csv'
def load_partial(name,fold):
    path=partial_path(name,fold)
    if not path.exists():return None
    f=pd.read_csv(path)
    assert set(['model','fold','isoform','canonical_smiles','y_true','y_pred','ci_low','ci_high']).issubset(f)
    assert (f.model==name).all() and (f.fold==fold).all()
    assert not f.duplicated(['model','fold','isoform','canonical_smiles']).any()
    valid=primary.loc[fold_ids==fold,'canonical_smiles']
    assert set(f.canonical_smiles).issubset(set(valid))
    print('Resuming',name,'fold',fold,'from',path)
    return f

def save_predictions(name,fold,test_idx,pred):
    assert pred.shape==(len(test_idx),4)
    records=[]
    for j,iso in enumerate(ISOFORMS):
        mask=np.isfinite(YR[test_idx,j])
        for idx,val in zip(test_idx[mask],pred[mask,j]):
            records.append(dict(model=name,fold=fold,isoform=iso,
                canonical_smiles=all_canon[idx],y_true=float(YR[idx,j]),y_pred=float(val),
                ci_low=float(CI_LOW[idx,j]),ci_high=float(CI_HIGH[idx,j])))
    frame=pd.DataFrame.from_records(records)
    assert len(frame)>0, f'No measured direct pIC50 in fold {fold}'
    path=partial_path(name,fold)
    tmp=path.with_suffix('.tmp')
    frame.to_csv(tmp,index=False);os.replace(tmp,path)
    return frame

leakage_records=[]
def assert_disjoint_groups(fold, fit_idx, dev_idx=None):
    test_idx=np.flatnonzero(fold_ids==fold)
    held=set(PRIMARY_G[test_idx]); fit_groups=set(ALL_G[fit_idx])
    assert not held.intersection(fit_groups), 'Outer heldout scaffold leaked into fit'
    if dev_idx is not None:
        assert not set(PRIMARY_G[dev_idx]).intersection(fit_groups), 'Inner dev scaffold leaked into fit'
    leakage_records.append(dict(fold=fold,n_fit=len(fit_idx),n_heldout=len(test_idx),
                                n_fit_extra=int((np.asarray(fit_idx)>=N_PRIMARY).sum()),
                                outer_scaffold_overlap=len(held.intersection(fit_groups)),
                                inner_scaffold_overlap=(len(set(PRIMARY_G[dev_idx]).intersection(fit_groups))
                                                        if dev_idx is not None else np.nan)))

# Training features are standardized and imputed only using eligible fitting structures.
# The fit pool and scaling are COMMON across ablation arms, so comparisons isolate labels.
def scaled_features(fit_idx):
    raw=X_RAW
    x_fit=raw[fit_idx]
    med=np.nanmedian(x_fit,axis=0)
    med[~np.isfinite(med)]=0
    filled=np.where(np.isfinite(raw),raw,med[None,:])
    scaler=StandardScaler().fit(filled[fit_idx])
    return np.clip(scaler.transform(filled),-8,8).astype(np.float32)

## 7 · Experiment A: controlled cross-assay supervision

**Ablation design** — the model architecture and input features are identical. Model configurations differ in which labels are enabled and whether auxiliary-only molecules join the fit pool:

| Arm | Direct pIC50 | Screening | Emax | TDI | Auxiliary-only molecules |
|---|:---:|:---:|:---:|:---:|:---:|
| `direct_only` | ✓ | | | | |
| `screen_only` | ✓ | ✓ | | | |
| `emax_only` | ✓ | | ✓ | | |
| `tdi_only` | ✓ | | | ✓ | |
| `all_primary` | ✓ | ✓ | ✓ | ✓ | |
| `screen_plus_extra` | ✓ | ✓ | | | ✓ |
| `all_plus_extra` | ✓ | ✓ | ✓ | ✓ | ✓ |

For `extra` arms, all auxiliary-only structures with *either* an outer-held-out or inner-development scaffold are excluded. Each family is given a fixed loss contribution, averaging over its observed tasks. Task-specific regression targets are standardized from training labels only. **No held-out experimental screening measurements are used as features.**

In [ ]:
ARM_CONFIG={
 'direct_only': (set(),False),
 'screen_only': ({'screen'},False),
 'emax_only': ({'emax'},False),
 'tdi_only': ({'tdi'},False),
 'all_primary': ({'screen','emax','tdi'},False),
 'screen_plus_extra': ({'screen'},True),
 'all_plus_extra': ({'screen','emax','tdi'},True),
}
assert all(a in ARM_CONFIG for a in ABLATION_ARMS)
family_col_indices={fam:[j for j,k in enumerate(reg_tasks) if j>=4 and k.startswith(fam+':')]
                    for fam in ['screen','emax','tdi']}
binary_tdi_indices=[j for j,k in enumerate(bin_tasks) if k.startswith('tdi:')]

class SmallAssayNet(tnn.Module):
    def __init__(self,n_inputs,n_reg,n_binary):
        super().__init__()
        self.net=tnn.Sequential(tnn.Linear(n_inputs,HIDDEN),tnn.LayerNorm(HIDDEN),
                    tnn.GELU(),tnn.Dropout(DROPOUT),
                    tnn.Linear(HIDDEN,HIDDEN//2),tnn.GELU(),tnn.Dropout(DROPOUT))
        self.reg=tnn.Linear(HIDDEN//2,n_reg)
        self.binary=tnn.Linear(HIDDEN//2,n_binary) if n_binary else None
    def forward(self,x):
        z=self.net(x)
        return self.reg(z),self.binary(z) if self.binary is not None else None

def masked_mean_per_task(loss,mask):
    counts=mask.sum(0)
    valid=counts>0
    if not valid.any():return loss.sum()*0
    return ((loss*mask).sum(0)/counts.clamp(min=1))[valid].mean()

def loss_fn(pr,pb,targets_reg,targets_bin,active_families):
    mask=torch.isfinite(targets_reg)
    target=torch.nan_to_num(targets_reg,nan=0.0)
    each=torch.nn.functional.smooth_l1_loss(pr,target,reduction='none',beta=1)
    loss=masked_mean_per_task(each[:,:4],mask[:,:4].float())
    for family,w in [('screen',0.20),('emax',0.10),('tdi',0.10)]:
        js=family_col_indices[family]
        if family in active_families and len(js):
            loss=loss+w*masked_mean_per_task(each[:,js],mask[:,js].float())
    if 'tdi' in active_families and pb is not None and binary_tdi_indices:
        observed=torch.isfinite(targets_bin[:,binary_tdi_indices])
        gt=torch.nan_to_num(targets_bin[:,binary_tdi_indices],nan=0.0)
        bce=torch.nn.functional.binary_cross_entropy_with_logits(pb[:,binary_tdi_indices],gt,reduction='none')
        loss=loss+0.08*masked_mean_per_task(bce,observed.float())
    return loss

def run_arm(fold,arm):
    active,extra_structures=ARM_CONFIG[arm]
    torch.manual_seed(SEED+fold);np.random.seed(SEED+fold)
    outer_train,outer_test,eligible=outer_split(fold)
    fit_primary,dev=dev_split(fold,outer_train)
    inner_groups=set(PRIMARY_G[dev])
    allowed_extra=np.array([i for i in eligible if i>=N_PRIMARY and ALL_G[i] not in inner_groups],dtype=int)
    # Exclude auxiliary-only structures without labels in any ACTIVE family.
    # Such structures would otherwise contribute zero loss yet change batch schedules.
    extra_active=np.zeros(N_ALL,dtype=bool)
    for fam in active:
        cols=family_col_indices[fam]
        if cols: extra_active |= np.isfinite(YR[:,cols]).any(axis=1)
    if 'tdi' in active and YB.shape[1]:
        extra_active |= np.isfinite(YB).any(axis=1)
    supervised_extra=allowed_extra[extra_active[allowed_extra]]
    fit_idx=np.r_[fit_primary,supervised_extra] if extra_structures else fit_primary
    assert_disjoint_groups(fold,fit_idx,dev)
    # Common (label-free) feature normalization pool across all model configurations.
    common_input_pool=np.r_[fit_primary,allowed_extra]
    X=scaled_features(common_input_pool)
    # All task normalizations are learned from eligible fit structures, never from dev/test labels.
    reg_mean=np.zeros(YR.shape[1],dtype=np.float32)
    reg_scale=np.ones(YR.shape[1],dtype=np.float32)
    for j in range(YR.shape[1]):
        vals=YR[common_input_pool,j];vals=vals[np.isfinite(vals)]
        if len(vals)>1:
            reg_mean[j]=np.mean(vals)
            reg_scale[j]=max(float(np.std(vals)),0.05)
    yn=(YR-reg_mean[None,:])/reg_scale[None,:]
    yn=yn.copy();binary=YB.copy()
    for fam,indices in family_col_indices.items():
        if fam not in active:yn[:,indices]=np.nan
    if 'tdi' not in active:binary[:]=np.nan
    data=TensorDataset(torch.from_numpy(X[fit_idx]),
        torch.from_numpy(yn[fit_idx]),torch.from_numpy(binary[fit_idx]))
    loader=DataLoader(data,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
    model=SmallAssayNet(X.shape[1],YR.shape[1],YB.shape[1]).to(DEVICE)
    optim=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    xdev=torch.from_numpy(X[dev]).to(DEVICE)
    ydev=YR[dev,:4];best=np.inf;best_state=None;bad=0;history=[]
    for epoch in range(MAX_EPOCHS):
        model.train();batch_losses=[]
        for xb,yr,yb in loader:
            xb=xb.to(DEVICE);yr=yr.to(DEVICE);yb=yb.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            pr,pb=model(xb)
            loss=loss_fn(pr,pb,yr,yb,active)
            if not torch.isfinite(loss):raise FloatingPointError(f'{arm} nonfinite loss')
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optim.step();batch_losses.append(float(loss.detach().cpu()))
        model.eval()
        with torch.no_grad():
            pred_dev=model(xdev)[0].detach().cpu().numpy()[:,:4]*reg_scale[:4]+reg_mean[:4]
        fold_maes=[np.mean(np.abs(ydev[np.isfinite(ydev[:,j]),j]-pred_dev[np.isfinite(ydev[:,j]),j]))
                   for j in range(4) if np.isfinite(ydev[:,j]).any()]
        score=float(np.mean(fold_maes)) if fold_maes else np.inf
        history.append(dict(model=arm,fold=fold,epoch=epoch,train_loss=float(np.mean(batch_losses)),
                            dev_macro_mae=score,n_fit=len(fit_idx)))
        if score<best-1e-4:
            best=score;bad=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:bad+=1
        if bad>=PATIENCE:break
    assert best_state is not None
    model.load_state_dict(best_state);model.to(DEVICE).eval()
    with torch.no_grad():
        p=model(torch.from_numpy(X[outer_test]).to(DEVICE))[0].detach().cpu().numpy()[:,:4]
    pred=p*reg_scale[:4]+reg_mean[:4]
    save_predictions('Ablation_'+arm,fold,outer_test,pred)
    # Keep auditable counts for each family in the actual supervised fit pool.
    counts={fam:int(np.isfinite(YR[np.ix_(fit_idx,indices)]).sum()) if indices else 0
            for fam,indices in family_col_indices.items()}
    counts['tdi_binary']=int(np.isfinite(YB[fit_idx]).sum())
    return history,dict(model=arm,fold=fold,n_primary_fit=len(fit_primary),
                        n_aux_only_fit=len(fit_idx)-len(fit_primary),best_dev_mae=best,**counts)

all_hist=[];ablation_fit_audit=[]
if RUN_ABLATIONS:
    for fold in FOLDS_TO_RUN:
        for arm in ABLATION_ARMS:
            if load_partial('Ablation_'+arm,fold) is not None:continue
            h,a=run_arm(fold,arm)
            all_hist+=h;ablation_fit_audit.append(a)
            pd.DataFrame(h).to_csv(PARTIAL/f'history_{arm}_fold{fold}.csv',index=False)
            pd.DataFrame([a]).to_csv(PARTIAL/f'fit_audit_{arm}_fold{fold}.csv',index=False)
            print('Completed',arm,'fold',fold,'dev MAE',round(a['best_dev_mae'],4))
history_files=sorted(PARTIAL.glob('history_*.csv'))
fit_files=sorted(PARTIAL.glob('fit_audit_*.csv'))
pd.concat([pd.read_csv(p) for p in history_files],ignore_index=True).to_csv(
    OUT/'training_history.csv',index=False) if history_files else pd.DataFrame().to_csv(OUT/'training_history.csv')
pd.concat([pd.read_csv(p) for p in fit_files],ignore_index=True).to_csv(
    OUT/'ablation_fit_audit.csv',index=False) if fit_files else pd.DataFrame().to_csv(OUT/'ablation_fit_audit.csv')

## 8 · Experiment B: cross-fitted screening → predicted pIC50

For each **outer** fold:

1. Split **outer-training primary compounds** into inner scaffold folds.
2. For each inner fold and each CYP, train a first-stage LightGBM screen-response predictor using screening measurements from **only other scaffolds**. Include eligible auxiliary-only structures only when they are outside all excluded groups.
3. Generate **inner out-of-fold** screening predictions for the original pIC50 training compounds. These four values become *training features* for a second-stage pIC50 model.
4. Refit the first-stage screen predictors on all eligible **outer-training** screening labels, and use them to generate screening features for the outer test molecules.
5. Train a second-stage LightGBM model using molecular features + OOF screening features. Compare with an **identical** LightGBM model using molecular features only, fitted on the same direct-inhibition rows.

The screening measurements for an outer-test structure or held-out scaffold are **never** used, even for the first-stage screen model. This is an honest test of a structure-predicted screening intermediate, not a test of observed screening readouts.

In [ ]:
# To reduce the inner cross-fitting cost, use RDKit + Morgan count + optional CheMeleon
# embeddings as the molecular features in BOTH stage-one and stage-two regressors.
# Note: LightGBM predictions from numpy arrays have identical train/predict column order.
SCREEN_TASKS=[next(t for t in reg_tasks if t.startswith(f'screen:{iso}:')) for iso in ISOFORMS]
SCREEN_Y=YR[:,[reg_tasks.index(t) for t in SCREEN_TASKS]]

def new_lgbm(seed):
    return lgb.LGBMRegressor(n_estimators=N_LGBM_TREES,learning_rate=.04,
               num_leaves=15,min_child_samples=12 if not FAST_MODE else 4,
               colsample_bytree=.90,subsample=.9,subsample_freq=1,
               reg_lambda=2.0,verbosity=-1,n_jobs=2,random_state=seed)

def fill_features(train_indices):
    raw=X_RAW
    med=np.nanmedian(raw[train_indices],axis=0)
    med[~np.isfinite(med)]=0
    return np.where(np.isfinite(raw),raw,med[None,:]).astype(np.float32)

def fit_screen_predictions(fold):
    primary_train,outer_test,eligible=outer_split(fold)
    n=len(primary_train)
    inner_groups=PRIMARY_G[primary_train]
    distinct=len(set(inner_groups))
    if distinct<2:raise ValueError('Not enough distinct training scaffolds for cross-fitting')
    n_inner=min(INNER_FOLDS,distinct)
    splitter=GroupKFold(n_splits=n_inner)
    screen_train_features=np.full((n,4),np.nan,dtype=np.float32)
    screen_test_features=np.full((len(outer_test),4),np.nan,dtype=np.float32)
    diagnostics=[];n_fit_log=[]
    # Screen-imputation preprocessing is also fitted only from training structures.
    full_X=fill_features(eligible)
    for subfold,(fit_pos,valid_pos) in enumerate(splitter.split(primary_train,groups=inner_groups)):
        val_idx=primary_train[valid_pos]
        val_groups=set(PRIMARY_G[val_idx])
        safe_idx=np.array([i for i in eligible if ALL_G[i] not in val_groups],dtype=int)
        assert_disjoint_groups(fold,safe_idx)
        assert not set(ALL_G[safe_idx]).intersection(val_groups)
        for j,iso in enumerate(ISOFORMS):
            observed=safe_idx[np.isfinite(SCREEN_Y[safe_idx,j])]
            if len(observed)>=8 and np.nanstd(SCREEN_Y[observed,j])>1e-8:
                model=new_lgbm(SEED+fold*100+subfold*10+j)
                model.fit(full_X[observed],SCREEN_Y[observed,j])
                pred=model.predict(full_X[val_idx])
            else:
                # Tiny synthetic test safeguard; explicit fallback from training labels only.
                val=SCREEN_Y[observed,j]
                pred=np.full(len(val_idx),float(np.mean(val)) if len(val) else 0.0)
            screen_train_features[valid_pos,j]=pred
            n_fit_log.append(dict(fold=fold,inner_fold=subfold,isoform=iso,
                                  n_screen_fit=len(observed),n_predicted=len(val_idx),
                                  inner_scaffold_overlap=len(set(ALL_G[observed]).intersection(val_groups)),
                                  outer_scaffold_overlap=len(set(ALL_G[observed]).intersection(set(PRIMARY_G[outer_test])))))
            screened=np.isfinite(SCREEN_Y[val_idx,j])
            if screened.any():
                e=metrics(SCREEN_Y[val_idx[screened],j],np.asarray(pred)[screened])
                diagnostics.append(dict(fold=fold,inner_fold=subfold,isoform=iso,
                                         mae=e['mae'],r2=e['r2'],n=e['n']))
    assert np.isfinite(screen_train_features).all()
    # Fit first-stage screen models on all eligible outer-training screen labels.
    for j,iso in enumerate(ISOFORMS):
        observed=eligible[np.isfinite(SCREEN_Y[eligible,j])]
        if len(observed)>=8 and np.nanstd(SCREEN_Y[observed,j])>1e-8:
            model=new_lgbm(SEED+fold*1000+j)
            model.fit(full_X[observed],SCREEN_Y[observed,j])
            screen_test_features[:,j]=model.predict(full_X[outer_test])
        else:
            val=SCREEN_Y[observed,j]
            screen_test_features[:,j]=float(np.mean(val)) if len(val) else 0.0
    assert np.isfinite(screen_test_features).all()
    return primary_train,outer_test,full_X,screen_train_features,screen_test_features,diagnostics,n_fit_log

stack_diagnostics=[];inner_fit_logs=[]
if RUN_SCREEN_STACK:
    for fold in FOLDS_TO_RUN:
        base_name='Direct_LGBM_control';stack_name='ScreenStack_LGBM'
        if (load_partial(base_name,fold) is not None and
            load_partial(stack_name,fold) is not None):continue
        tr,te,full_X,p_screen_oof,p_screen_test,d,logs=fit_screen_predictions(fold)
        stack_diagnostics.extend(d);inner_fit_logs.extend(logs)
        pd.DataFrame(d).to_csv(PARTIAL/f'screen_diag_fold{fold}.csv',index=False)
        pd.DataFrame(logs).to_csv(PARTIAL/f'screen_fit_fold{fold}.csv',index=False)
        # Second-stage training features have *OOF* screen predictions, not observed responses.
        # The control is trained on exactly the same original molecular features and rows.
        assert p_screen_oof.shape==(len(tr),4)
        assert p_screen_test.shape==(len(te),4)
        xtr0=full_X[tr];xte0=full_X[te]
        xtr1=np.concatenate([xtr0,p_screen_oof],axis=1)
        xte1=np.concatenate([xte0,p_screen_test],axis=1)
        pred_base=np.full((len(te),4),np.nan,dtype=np.float32)
        pred_stack=np.full_like(pred_base,np.nan)
        for j,iso in enumerate(ISOFORMS):
            observed=np.isfinite(YR[tr,j]);n=int(observed.sum())
            if n<8:raise ValueError(f'Too few {iso} direct measurements in outer training')
            model0=new_lgbm(SEED+fold*10+j)
            model1=new_lgbm(SEED+fold*10+j)
            model0.fit(xtr0[observed],YR[tr[observed],j]);pred_base[:,j]=model0.predict(xte0)
            model1.fit(xtr1[observed],YR[tr[observed],j]);pred_stack[:,j]=model1.predict(xte1)
        save_predictions(base_name,fold,te,pred_base)
        save_predictions(stack_name,fold,te,pred_stack)
        print('Finished nested screen transfer fold',fold)
diag_parts=sorted(PARTIAL.glob('screen_diag_fold*.csv'))
fit_parts=sorted(PARTIAL.glob('screen_fit_fold*.csv'))
(pd.concat([pd.read_csv(p) for p in diag_parts],ignore_index=True) if diag_parts else pd.DataFrame()).to_csv(
    OUT/'screen_inner_oof_diagnostics.csv',index=False)
(pd.concat([pd.read_csv(p) for p in fit_parts],ignore_index=True) if fit_parts else pd.DataFrame()).to_csv(
    OUT/'screen_inner_fit_counts.csv',index=False)
if fit_parts:
    audit=pd.read_csv(OUT/'screen_inner_fit_counts.csv')
    assert (audit.inner_scaffold_overlap==0).all() and (audit.outer_scaffold_overlap==0).all()

## 9 · Compare ablation arms, isoforms and five-fold out-of-fold predictions

In [ ]:
files=sorted(p for p in PARTIAL.glob('*_fold*.csv')
             if p.name.startswith(('Ablation_', 'Direct_LGBM_control_', 'ScreenStack_LGBM_')))
if not files:raise RuntimeError('No completed model predictions; enable an experiment above')
predictions=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
assert not predictions.duplicated(['model','fold','isoform','canonical_smiles']).any()
predictions.to_csv(OUT/'predictions.csv',index=False)
rows=[]
for (model,fold,iso),sub in predictions.groupby(['model','fold','isoform']):
    rows.append(dict(model=model,fold=int(fold),isoform=iso,
        **metrics(sub.y_true,sub.y_pred,sub.ci_low,sub.ci_high)))
fold_metrics=pd.DataFrame(rows)
fold_metrics.to_csv(OUT/'fold_metrics.csv',index=False)
summary=[]
for (model,iso),sub in predictions.groupby(['model','isoform']):
    summary.append(dict(model=model,isoform=iso,
        **metrics(sub.y_true,sub.y_pred,sub.ci_low,sub.ci_high)))
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'summary.csv',index=False)
macro=summary.groupby('model',as_index=False)[['mae','mean_interval_excess','spearman']].mean()
macro=macro.sort_values('mae')
macro.to_csv(OUT/'macro_summary.csv',index=False)
display(macro)
display(summary.pivot(index='model',columns='isoform',values='mae'))

# The main controlled comparisons; all members here use the same exact direct labels.
ref=summary.set_index(['model','isoform']).mae
ablation=[]
for arm in ABLATION_ARMS:
    m='Ablation_'+arm
    for iso in ISOFORMS:
        if ('Ablation_direct_only',iso) in ref and (m,iso) in ref:
            ablation.append(dict(arm=arm,isoform=iso,baseline_mae=ref['Ablation_direct_only',iso],
               arm_mae=ref[m,iso],improvement=ref['Ablation_direct_only',iso]-ref[m,iso]))
abl=pd.DataFrame(ablation)
abl.to_csv(OUT/'auxiliary_ablation.csv',index=False)
display(abl)

if 'Direct_LGBM_control' in set(predictions.model) and 'ScreenStack_LGBM' in set(predictions.model):
    control=predictions[predictions.model=='Direct_LGBM_control']
    stack=predictions[predictions.model=='ScreenStack_LGBM']
    key=['fold','isoform','canonical_smiles']
    joined=control.merge(stack,on=key,suffixes=('_direct','_stack'),validate='one_to_one')
    assert len(joined)==len(control)==len(stack)
    assert np.allclose(joined.y_true_direct,joined.y_true_stack)
    compare=[]
    for iso,sub in joined.groupby('isoform'):
        before=mean_absolute_error(sub.y_true_direct,sub.y_pred_direct)
        after=mean_absolute_error(sub.y_true_stack,sub.y_pred_stack)
        compare.append(dict(isoform=iso,direct_mae=before,stacked_mae=after,
                            improvement=before-after,n=len(sub)))
    compare=pd.DataFrame(compare)
    compare.to_csv(OUT/'screen_to_pic50_comparison.csv',index=False)
    display(compare)

## 10 · Plots and leakage checks

In [ ]:
plt.figure(figsize=(11,4.8))
models=macro.model.tolist();values=macro.mae.to_numpy()
plt.barh(models[::-1],values[::-1]);plt.xlabel('Macro scaffold OOF MAE (pIC50 units)')
plt.title('Notebook 08: direct-inhibition model comparison')
plt.tight_layout();plt.savefig(OUT/'01_macro_model_comparison.png',dpi=160);plt.show()
if len(abl):
    tab=abl.pivot(index='arm',columns='isoform',values='improvement')
    fig,ax=plt.subplots(figsize=(9,5));
    im=ax.imshow(tab.values,aspect='auto',cmap='coolwarm',vmin=-max(.01,np.nanmax(abs(tab.values))),
                 vmax=max(.01,np.nanmax(abs(tab.values))))
    ax.set_xticks(range(len(tab.columns)),tab.columns)
    ax.set_yticks(range(len(tab.index)),tab.index)
    ax.set_title('Auxiliary learning: positive values mean lower MAE')
    plt.colorbar(im,ax=ax,label='Direct-only MAE minus auxiliary MAE')
    plt.tight_layout();plt.savefig(OUT/'02_auxiliary_ablation_heatmap.png',dpi=160);plt.show()

pd.DataFrame(leakage_records).to_csv(OUT/'leakage_and_split_audit.csv',index=False)
if leakage_records:
    assert all(x['outer_scaffold_overlap']==0 for x in leakage_records)
    assert all(pd.isna(x['inner_scaffold_overlap']) or x['inner_scaffold_overlap']==0
               for x in leakage_records)
print('Scaffold-leakage checks passed.')

## 11 · Reproducibility summary and downloadable ZIP

In [ ]:
metadata=dict(notebook='08_screening_multifidelity_ablations.ipynb',
    inputs_sha256=source_shas,run_signature=RUN_SIGNATURE,experiment_tag=EXPERIMENT_TAG,
    fast_mode=FAST_MODE,folds_completed=FOLDS_TO_RUN,n_splits=N_SPLITS,
    full_oof=sorted(FOLDS_TO_RUN)==list(range(N_SPLITS)),
    sample_counts=dict(primary=N_PRIMARY,union=N_ALL,aux_only=len(extra)),
    feature_dimensions=dict(total=X_RAW.shape[1],chemeleon=emb.shape[1],rdkit=desc.shape[1],morgan=count.shape[1]),
    embedding_type='mock_synthetic_only' if MOCK_EMBEDDINGS else 'pretrained' if USE_CHEMELEON else 'none',
    screen_columns=dict(iso=SCREEN_ISO_COL,value=SCREEN_VALUE_COL),
    exact_competition_metric_implemented=False,
    note='All measurements for test/inner-dev scaffolds excluded from multi-assay fitting; screen stacking uses inner grouped OOF screening predictions.',
    settings=setting_fields,python=platform.python_version(),torch=torch.__version__,
    rdkit=rdBase.rdkitVersion,lightgbm=lgb.__version__)
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
archive=BASE/'cyp_notebook_08_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob('*')):
        if p.is_file():z.write(p,'cyp_notebook_08_results/'+p.name)
print('Generated:',archive,'bytes',archive.stat().st_size)
print('Exported files:',[p.name for p in OUT.glob('*') if p.is_file()])
try:
    from IPython.display import FileLink,display
    display(FileLink(str(archive)))
except Exception: pass

## 12 · Interpretation checklist

**Do not report these numbers as the official challenge score.** The interval-excess diagnostic is not MA-ST-RAE, and the model-selection process may have seen results from earlier notebooks. For a publishable claim, repeat hyperparameter selection inside the outer training data or use an additional independent holdout.

Questions to answer:

- Is `Ablation_screen_only` consistently better than `Ablation_direct_only` for *some* CYPs?
- Does adding Emax or TDI induce negative transfer? Are improvements reproducible across outer folds?
- Does `all_primary` differ from `all_plus_extra`? The comparison isolates extra labeled chemical coverage (but the primary/extra sampling mechanisms differ).
- Does `ScreenStack_LGBM` beat `Direct_LGBM_control` on **identical** fold/compound sets?
- Does the first-stage screening model achieve reasonable inner OOF predictive accuracy? If not, why would downstream predicted-screening features help?
- Confirm the `screen` audit has exactly four tasks, each with the expected response column.
- Audit `leakage_and_split_audit.csv`, `screen_inner_fit_counts.csv`, fold identities, and run signature.
- Use **`FAST_MODE=False`** before drawing quantitative conclusions from the full dataset.

**Future research:** (i) assay-condition embeddings rather than independent task heads; (ii) learned uncertainty/propensity models for follow-up selection; (iii) isoform-conditioned mixture-of-experts; (iv) nested evaluation of adaptive blends. These would require additional experiments, not claims derived from this notebook.